In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Train Global XGBoost Model (All Years) & Generate Diagnostics
-- Trains one unified model to learn global fire rules --
-- Outputs Feature Importance and Partial Dependence Plots (PDPs) --
"""

import os
import sys
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import pyarrow.dataset as ds
from sklearn.model_selection import train_test_split
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

# Set non-interactive backend for server environments
plt.switch_backend('Agg')

# ============================================================
# CONFIG
# ============================================================

RANDOM_STATE = 42
DATASET_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/parquet_cems_with_fraction_dataset_burnedlab_mask_analytical")

# OUTPUT DIRECTORY for Diagnostics
OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_global_diagnostics")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUT_DIR / "global_training_log.txt"

FEATURES = [
    "DEM", "slope", "aspect", "b1", "relative_humidity",
    "total_precipitation_sum", "temperature_2m", "temperature_2m_min",
    "temperature_2m_max", "build_up_index", "drought_code",
    "duff_moisture_code", "fine_fuel_moisture_code",
    "fire_weather_index", "initial_fire_spread_index",
]

FRACTION_COL = "fraction"
LABEL_COL = "burned"
VAL_SIZE = 0.20 

# TRAINING CONFIG
FINAL_ROUNDS  = 3000            
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", "0")) or os.cpu_count() or 8
USE_GPU = bool(os.environ.get("CUDA_VISIBLE_DEVICES", "").strip())
TREE_METHOD = "hist" # Standardizing on hist for broader compatibility with sklearn PDPs

# ============================================================
# HELPERS
# ============================================================

class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w", encoding='utf-8')
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    def flush(self):
        self.terminal.flush()
        self.log.flush()

def prepare_df_cleaned(df: pd.DataFrame):
    df = df.copy()
    
    # 1. Types & Fraction
    df[FRACTION_COL] = pd.to_numeric(df[FRACTION_COL], errors="coerce").astype("float32")
    df = df[df[FRACTION_COL].notna() & (df[FRACTION_COL] != 0.5)].copy()
    df[LABEL_COL] = (df[FRACTION_COL] > 0.5).astype("uint8")
    
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["b1"] = pd.to_numeric(df["b1"], errors="coerce").round().astype("Int64")

    # 2. Filter Negative FWI
    fwi_cols = ["duff_moisture_code", "drought_code", "fine_fuel_moisture_code", "build_up_index"]
    for c in fwi_cols:
        if c in df.columns:
            df = df[df[c] >= 0]

    # 3. NaNs
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=FEATURES + [LABEL_COL])
    
    for c in FEATURES:
        if c == "b1": continue
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")
    
    df["b1"] = df["b1"].astype("category")
        
    return df.copy()

def load_data():
    print(f"Loading Dataset from: {DATASET_DIR}")
    dset = ds.dataset(str(DATASET_DIR), format="parquet", partitioning="hive")
    cols = FEATURES + [FRACTION_COL, "year"]
    available_cols = dset.schema.names
    cols_to_load = [c for c in cols if c in available_cols]
    
    table = dset.to_table(columns=cols_to_load)
    df = table.to_pandas()
    return prepare_df_cleaned(df)

# ============================================================
# MAIN LOOP
# ============================================================

def main():
    sys.stdout = Logger(str(LOG_FILE))
    print("=" * 60)
    print("STARTING GLOBAL DIAGNOSTIC TRAINING")
    print("=" * 60)
    
    # 1. Load Data
    df_all = load_data()
    print(f"Total Rows Loaded: {len(df_all):,}")
    
    X = df_all[FEATURES]
    y = df_all[LABEL_COL].values
    
    # 2. Split Data (No test set, just train/val for early stopping)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=VAL_SIZE, 
        random_state=RANDOM_STATE, stratify=y
    )

    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    scale_weight = n_neg / max(1, n_pos)
    
    print(f"Train: {len(X_train):,} | Val: {len(X_val):,}")
    print(f"Scale Pos Weight: {scale_weight:.2f}")

    # 3. Train Global Model using Scikit-Learn API
    model = xgb.XGBClassifier(
        n_estimators=FINAL_ROUNDS,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method=TREE_METHOD,
        max_bin=256,
        seed=RANDOM_STATE,
        learning_rate=0.05,
        scale_pos_weight=scale_weight,
        max_depth=4,            
        min_child_weight=100,  
        gamma=5.0,              
        subsample=0.5,          
        colsample_bytree=0.5,  
        reg_lambda=10.0,        
        reg_alpha=1.0, 
        n_jobs=N_JOBS,
        enable_categorical=True,
        early_stopping_rounds=200
    )

    print("\nTraining Global Model...")
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )
    
    # Save the global model
    model_path = OUT_DIR / "xgb_global_model.json"
    model.save_model(model_path)
    print(f"\nModel saved to: {model_path}")

    # ==========================================
    # DIAGNOSTICS: FEATURE IMPORTANCE
    # ==========================================
    print("\nGenerating Feature Importance Plot...")
    fig, ax = plt.subplots(figsize=(10, 8))
    xgb.plot_importance(model, ax=ax, importance_type='gain', max_num_features=15, title='Feature Importance (Gain)')
    plt.tight_layout()
    plt.savefig(OUT_DIR / "feature_importance_gain.png", dpi=300)
    plt.close()

    # ==========================================
    # DIAGNOSTICS: PARTIAL DEPENDENCE PLOTS
    # ==========================================
    print("Generating Partial Dependence Plots...")
    
    # Subsample data to compute PDPs quickly (50k rows is plenty for a smooth curve)
    sample_size = min(50000, len(X_train))
    X_sample = X_train.sample(n=sample_size, random_state=RANDOM_STATE)
    
    # Isolate continuous features (sklearn PDP struggles with categorical 'b1' natively)
    continuous_features = [f for f in FEATURES if f != "b1"]
    
    # Get top 6 most important continuous features to plot
    importance_dict = model.get_booster().get_score(importance_type='gain')
    sorted_features = sorted(importance_dict.keys(), key=lambda k: importance_dict[k], reverse=True)
    top_continuous = [f for f in sorted_features if f in continuous_features][:6]
    
    print(f"Plotting PDPs for top continuous features: {top_continuous}")

    fig, ax = plt.subplots(figsize=(15, 10))
    PartialDependenceDisplay.from_estimator(
        estimator=model,
        X=X_sample,
        features=top_continuous,
        grid_resolution=50,
        ax=ax,
        n_cols=3
    )
    plt.suptitle("Partial Dependence Plots (Top 6 Features)", fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "partial_dependence_plots.png", dpi=300, bbox_inches='tight')
    plt.close()

    print(f"All diagnostics saved to {OUT_DIR}")

if __name__ == "__main__":
    main()

STARTING GLOBAL DIAGNOSTIC TRAINING
Loading Dataset from: /explore/nobackup/people/spotter5/clelland_fire_ml/parquet_cems_with_fraction_dataset_burnedlab_mask_analytical
Total Rows Loaded: 75,674,335
Train: 60,539,468 | Val: 15,134,867
Scale Pos Weight: 321.76

Training Global Model...
[0]	validation_0-logloss:0.66929
[100]	validation_0-logloss:0.39477
[200]	validation_0-logloss:0.37064
[300]	validation_0-logloss:0.35796
[400]	validation_0-logloss:0.35084
[500]	validation_0-logloss:0.34560
[600]	validation_0-logloss:0.34123
[700]	validation_0-logloss:0.33728
[800]	validation_0-logloss:0.33397
[900]	validation_0-logloss:0.33086
[1000]	validation_0-logloss:0.32813
[1100]	validation_0-logloss:0.32572
[1200]	validation_0-logloss:0.32325
[1300]	validation_0-logloss:0.32077
[1400]	validation_0-logloss:0.31883
[1500]	validation_0-logloss:0.31698
[1600]	validation_0-logloss:0.31498
[1700]	validation_0-logloss:0.31322
[1800]	validation_0-logloss:0.31160
[1900]	validation_0-logloss:0.30999
[2000

SHAP plots

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Train Global XGBoost Model (All Years) & Generate Diagnostics
-- Trains one unified model to learn global fire rules --
-- Outputs Feature Importance, Partial Dependence Plots, and SHAP plots --
"""

import os
import sys
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import pyarrow.dataset as ds
from sklearn.model_selection import train_test_split
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
import shap  # <--- NEW: Import SHAP

# Set non-interactive backend for server environments
plt.switch_backend('Agg')

# ============================================================
# CONFIG
# ============================================================

RANDOM_STATE = 42
DATASET_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/parquet_cems_with_fraction_dataset_burnedlab_mask_analytical")

# OUTPUT DIRECTORY for Diagnostics
OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_global_diagnostics")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUT_DIR / "global_training_log.txt"

FEATURES = [
    "DEM", "slope", "aspect", "b1", "relative_humidity",
    "total_precipitation_sum", "temperature_2m", "temperature_2m_min",
    "temperature_2m_max", "build_up_index", "drought_code",
    "duff_moisture_code", "fine_fuel_moisture_code",
    "fire_weather_index", "initial_fire_spread_index",
]

FRACTION_COL = "fraction"
LABEL_COL = "burned"
VAL_SIZE = 0.20 

# TRAINING CONFIG
FINAL_ROUNDS  = 3000            
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", "0")) or os.cpu_count() or 8
USE_GPU = bool(os.environ.get("CUDA_VISIBLE_DEVICES", "").strip())
TREE_METHOD = "hist" # Standardizing on hist for broader compatibility with sklearn PDPs

# ============================================================
# HELPERS
# ============================================================

class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w", encoding='utf-8')
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    def flush(self):
        self.terminal.flush()
        self.log.flush()

def prepare_df_cleaned(df: pd.DataFrame):
    df = df.copy()
    
    # 1. Types & Fraction
    df[FRACTION_COL] = pd.to_numeric(df[FRACTION_COL], errors="coerce").astype("float32")
    df = df[df[FRACTION_COL].notna() & (df[FRACTION_COL] != 0.5)].copy()
    df[LABEL_COL] = (df[FRACTION_COL] > 0.5).astype("uint8")
    
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["b1"] = pd.to_numeric(df["b1"], errors="coerce").round().astype("Int64")

    # 2. Filter Negative FWI
    fwi_cols = ["duff_moisture_code", "drought_code", "fine_fuel_moisture_code", "build_up_index"]
    for c in fwi_cols:
        if c in df.columns:
            df = df[df[c] >= 0]

    # 3. NaNs
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=FEATURES + [LABEL_COL])
    
    for c in FEATURES:
        if c == "b1": continue
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")
    
    df["b1"] = df["b1"].astype("category")
        
    return df.copy()

def load_data():
    print(f"Loading Dataset from: {DATASET_DIR}")
    dset = ds.dataset(str(DATASET_DIR), format="parquet", partitioning="hive")
    cols = FEATURES + [FRACTION_COL, "year"]
    available_cols = dset.schema.names
    cols_to_load = [c for c in cols if c in available_cols]
    
    table = dset.to_table(columns=cols_to_load)
    df = table.to_pandas()
    return prepare_df_cleaned(df)

# ============================================================
# MAIN LOOP
# ============================================================

def main():
    sys.stdout = Logger(str(LOG_FILE))
    print("=" * 60)
    print("STARTING GLOBAL DIAGNOSTIC TRAINING")
    print("=" * 60)
    
    # 1. Load Data
    df_all = load_data()
    print(f"Total Rows Loaded: {len(df_all):,}")
    
    X = df_all[FEATURES]
    y = df_all[LABEL_COL].values
    
    # 2. Split Data (No test set, just train/val for early stopping)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=VAL_SIZE, 
        random_state=RANDOM_STATE, stratify=y
    )

    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    scale_weight = n_neg / max(1, n_pos)
    
    print(f"Train: {len(X_train):,} | Val: {len(X_val):,}")
    print(f"Scale Pos Weight: {scale_weight:.2f}")

    # 3. Train Global Model using Scikit-Learn API
    model = xgb.XGBClassifier(
        n_estimators=FINAL_ROUNDS,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method=TREE_METHOD,
        max_bin=256,
        seed=RANDOM_STATE,
        learning_rate=0.05,
        scale_pos_weight=scale_weight,
        max_depth=4,            
        min_child_weight=100,  
        gamma=5.0,              
        subsample=0.5,          
        colsample_bytree=0.5,  
        reg_lambda=10.0,        
        reg_alpha=1.0, 
        n_jobs=N_JOBS,
        enable_categorical=True,
        early_stopping_rounds=200
    )

    print("\nTraining Global Model...")
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )
    
    # Save the global model
    model_path = OUT_DIR / "xgb_global_model.json"
    model.save_model(model_path)
    print(f"\nModel saved to: {model_path}")

    # ==========================================
    # DIAGNOSTICS: FEATURE IMPORTANCE
    # ==========================================
    print("\nGenerating Feature Importance Plot...")
    fig, ax = plt.subplots(figsize=(10, 8))
    xgb.plot_importance(model, ax=ax, importance_type='gain', max_num_features=15, title='Feature Importance (Gain)')
    plt.tight_layout()
    plt.savefig(OUT_DIR / "feature_importance_gain.png", dpi=300)
    plt.close()

    # ==========================================
    # DIAGNOSTICS: PARTIAL DEPENDENCE PLOTS
    # ==========================================
    print("Generating Partial Dependence Plots...")
    
    # Subsample data to compute PDPs quickly (50k rows is plenty for a smooth curve)
    sample_size = min(50000, len(X_train))
    X_sample = X_train.sample(n=sample_size, random_state=RANDOM_STATE)
    
    # Isolate continuous features (sklearn PDP struggles with categorical 'b1' natively)
    continuous_features = [f for f in FEATURES if f != "b1"]
    
    # Get top 6 most important continuous features to plot
    importance_dict = model.get_booster().get_score(importance_type='gain')
    sorted_features = sorted(importance_dict.keys(), key=lambda k: importance_dict[k], reverse=True)
    top_continuous = [f for f in sorted_features if f in continuous_features][:6]
    
    print(f"Plotting PDPs for top continuous features: {top_continuous}")

    fig, ax = plt.subplots(figsize=(15, 10))
    PartialDependenceDisplay.from_estimator(
        estimator=model,
        X=X_sample,
        features=top_continuous,
        grid_resolution=50,
        ax=ax,
        n_cols=3
    )
    plt.suptitle("Partial Dependence Plots (Top 6 Features)", fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "partial_dependence_plots.png", dpi=300, bbox_inches='tight')
    plt.close()

    # ==========================================
    # DIAGNOSTICS: SHAP VALUES
    # ==========================================
    print("\nGenerating SHAP Summary Plot...")
    
    # Sample 10,000 rows for SHAP to prevent memory overload
    shap_sample_size = min(10000, len(X_train))
    X_shap = X_train.sample(n=shap_sample_size, random_state=RANDOM_STATE)

    # SHAP requires the underlying booster when using categorical features in XGBoost
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)

    # Create the Summary Beeswarm Plot
    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_values, X_shap, show=False)
    plt.title("SHAP Summary Plot", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / "shap_summary_plot.png", dpi=300, bbox_inches='tight')
    plt.close()

    print(f"All diagnostics saved to {OUT_DIR}")

if __name__ == "__main__":
    main()

/home/spotter5/.conda/envs/xgboost_gpu/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


STARTING GLOBAL DIAGNOSTIC TRAINING
Loading Dataset from: /explore/nobackup/people/spotter5/clelland_fire_ml/parquet_cems_with_fraction_dataset_burnedlab_mask_analytical


In [ ]:
't'

Determine pixels burning multiple times and why.  Use histograms. 

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Pixel Autopsy: Analyze Features of Hyper-Active Burn Pixels
-- Compares static terrain/land cover of pixels that burn repeatedly vs rarely --
-- Generates boxplots and histograms for diagnostics --
"""

import os
import numpy as np
import pandas as pd
import rasterio as rio
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# Set non-interactive backend
plt.switch_backend('Agg')

# ============================
# CONFIG
# ============================

PRED_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/predictions_loyo_regularized/annual_binary_masks")
FEATURE_TIF_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction")

# Output for our diagnostic plots
OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_global_diagnostics/autopsy")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Band indices in the feature TIFFs based on your FEATURES list (1-indexed for Rasterio)
# 1: DEM, 2: slope, 3: aspect, 4: b1
STATIC_BANDS = {
    "DEM": 1,
    "Slope": 2,
    "Aspect": 3,
    "LandCover_b1": 4
}

# ============================
# MAIN
# ============================

def main():
    print("=" * 60)
    print("STARTING PIXEL AUTOPSY")
    print("=" * 60)

    # 1. Accumulate Burn Counts
    pred_files = sorted(list(PRED_DIR.glob("loyo_regularized_annual_binary_*.tif")))
    if not pred_files:
        print("No prediction files found. Exiting.")
        return

    print(f"Aggregating {len(pred_files)} years of predictions...")
    cumulative_burn = None
    
    for pred_path in tqdm(pred_files, desc="Summing burns"):
        with rio.open(pred_path) as src:
            mask = src.read(1).astype(np.int32) 
            if cumulative_burn is None:
                cumulative_burn = mask
            else:
                cumulative_burn += mask

    # 2. Extract Static Features from ONE representative TIFF
    # We just need one because DEM, Slope, Aspect, and b1 do not change month-to-month
    sample_tif = list(FEATURE_TIF_DIR.glob("*_with_fraction.tif"))[0]
    print(f"\nExtracting static features from: {sample_tif.name}")

    with rio.open(sample_tif) as src:
        nodata_val = src.nodata
        
        # Read the arrays
        dem = src.read(STATIC_BANDS["DEM"]).astype('float32')
        slope = src.read(STATIC_BANDS["Slope"]).astype('float32')
        aspect = src.read(STATIC_BANDS["Aspect"]).astype('float32')
        b1 = src.read(STATIC_BANDS["LandCover_b1"]).astype('float32')

    # 3. Flatten and Clean the Data
    print("Flattening spatial data into a DataFrame...")
    df = pd.DataFrame({
        "Burn_Count": cumulative_burn.flatten(),
        "DEM": dem.flatten(),
        "Slope": slope.flatten(),
        "Aspect": aspect.flatten(),
        "LandCover_b1": b1.flatten()
    })

    # Drop NoData/Ocean pixels (where DEM is usually missing or extreme negative)
    initial_len = len(df)
    if nodata_val is not None:
        df = df[df["DEM"] != nodata_val]
    df = df[df["DEM"] > -500]  # Standard filter for valid land elevations
    df = df.dropna()
    print(f"Filtered out NoData/Ocean. Valid land pixels: {len(df):,} (from {initial_len:,})")

    # 4. Create Burn Frequency Categories
    def categorize_burns(count):
        if count == 0: return "0: Never Burned"
        elif count <= 5: return "1-5 Times"
        elif count <= 10: return "6-10 Times"
        else: return "11+ Times (Hyper-Active)"

    df["Burn_Category"] = df["Burn_Count"].apply(categorize_burns)
    
    # Sort for consistent plot ordering
    category_order = ["0: Never Burned", "1-5 Times", "6-10 Times", "11+ Times (Hyper-Active)"]

    print("\nPixel Distribution by Category:")
    print(df["Burn_Category"].value_counts())

    # 5. Plotting: Boxplots for Continuous Variables
    print("\nGenerating Diagnostic Boxplots...")
    sns.set_theme(style="whitegrid")

    continuous_vars = ["DEM", "Slope", "Aspect"]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for i, var in enumerate(continuous_vars):
        sns.boxplot(
            data=df, 
            x="Burn_Category", 
            y=var, 
            ax=axes[i], 
            order=category_order,
            palette="Reds"
        )
        axes[i].set_title(f"Distribution of {var} by Burn Frequency", fontweight='bold')
        axes[i].set_xlabel("")
        axes[i].set_ylabel(var)
        axes[i].tick_params(axis='x', rotation=15)

    plt.tight_layout()
    plt.savefig(OUT_DIR / "static_feature_boxplots.png", dpi=300)
    plt.close()

    # 6. Plotting: Histogram/Bar Chart for Categorical Land Cover (b1)
    print("Generating Land Cover (b1) distribution...")
    
    # Calculate percentage of each land cover type within the hyper-active group vs never burned
    # To see if hyper-active pixels are exclusively one specific land cover
    plt.figure(figsize=(12, 6))
    sns.histplot(
        data=df[df["Burn_Category"].isin(["0: Never Burned", "11+ Times (Hyper-Active)"])],
        x="LandCover_b1",
        hue="Burn_Category",
        stat="probability",
        common_norm=False,
        multiple="dodge",
        palette=["#cccccc", "#cc0000"],
        bins=30
    )
    plt.title("Land Cover (b1) Proportions: Hyper-Active vs Never Burned", fontsize=14, fontweight='bold')
    plt.xlabel("Land Cover Class (b1)")
    plt.ylabel("Proportion of Pixels in Category")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "landcover_b1_distribution.png", dpi=300)
    plt.close()

    print(f"\n✅ Pixel Autopsy complete. Plots saved to: {OUT_DIR}")

if __name__ == "__main__":
    main()

Summing burns: 100%|██████████| 22/22 [00:00<00:00, 25.72it/s]
/explore/nobackup/people/spotter5/temp_dir/ipykernel_2375655/2554882072.py:124: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/explore/nobackup/people/spotter5/temp_dir/ipykernel_2375655/2554882072.py:124: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/explore/nobackup/people/spotter5/temp_dir/ipykernel_2375655/2554882072.py:124: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Pixel Autopsy: Analyze Features of Frequently Burned Pixels
-- Computes the ALL-YEARS mean for dynamic weather and FWI features --
-- Maps numerical Land Cover (b1) values to descriptive labels --
-- Generates comprehensive diagnostic boxplots and bar charts --
"""

import os
import numpy as np
import pandas as pd
import rasterio as rio
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# Set non-interactive backend
plt.switch_backend('Agg')

# ============================
# CONFIG
# ============================

# Point to the original dynamic LOYO predictions (NOT the 0.90 ones)
PRED_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_loyo_regularized_new_fwi/annual_binary_masks")

# Point to the new FWI feature directory
FEATURE_TIF_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_new_fwi_with_fraction")

# Output directory specifically for the new FWI autopsy
OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_global_diagnostics/autopsy_new_fwi")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Feature Band Indices (1-indexed for Rasterio)
STATIC_FEATURES = {
    "DEM": 1,
    "Slope": 2,
    "Aspect": 3,
    "LandCover_b1": 4
}

DYNAMIC_FEATURES = {
    "Relative Humidity": 5,
    "Total Precip Sum": 6,
    "Temp 2m": 7,
    "Temp 2m Min": 8,
    "Temp 2m Max": 9,
    "Build Up Index": 10,
    "Drought Code": 11,
    "Duff Moisture Code": 12,
    "Fine Fuel Moisture Code": 13,
    "Fire Weather Index": 14,
    "Initial Fire Spread Index": 15
}

# Transcribed Land Cover Dictionary
LAND_COVER_DICT = {
    0: "Unclassified", 6: "Poplar forest", 7: "Oak forest", 10: "Pine forest",
    11: "Fir forest", 12: "Aspen forest", 13: "Hemlock forest", 14: "Spruce forest",
    15: "White Spruce", 16: "Black Spruce", 17: "Mixed forest", 18: "Larch forest",
    19: "Birch forest", 20: "Scotts Pine forest", 21: "Jack Pine forest",
    22: "Cedar forest", 23: "Maple", 24: "Siberian Pine", 25: "Linden",
    26: "Cedar Elfin Wood", 30: "Alpine shrubland", 31: "Riparian shrubland",
    32: "Other shrubland", 33: "Herbaceous", 35: "Sparsely Vegetated",
    50: "Bog", 51: "Fen", 52: "Marsh", 60: "Erect-shrub tundra",
    61: "Prostrate-shrub tundra", 62: "Shrub tundra", 63: "Graminoid tundra",
    64: "Wet-sedge tundra", 65: "Barren tundra", 70: "Developed",
    98: "Snow/Ice", 99: "Other"
}

# ============================
# MAIN
# ============================

def main():
    print("=" * 60)
    print("STARTING PIXEL AUTOPSY (NEW FWI - DYNAMIC THRESHOLDS)")
    print("=" * 60)

    # 1. Accumulate Burn Counts
    pred_files = sorted(list(PRED_DIR.glob("loyo_regularized_new_fwi_annual_binary_*.tif")))
    
    if not pred_files:
        print(f"❌ No prediction files found in {PRED_DIR}. Exiting.")
        return

    # Extract year cleanly
    years = [int(f.stem.split('_')[-1]) for f in pred_files]
    print(f"Aggregating {len(pred_files)} years of predictions...")
    cumulative_burn = None
    
    for pred_path in tqdm(pred_files, desc="Summing burns"):
        with rio.open(pred_path) as src:
            mask = src.read(1).astype(np.int32) 
            if cumulative_burn is None:
                cumulative_burn = mask
            else:
                cumulative_burn += mask

    # 2. Extract Static Features & Accumulate Dynamic Features across ALL years
    print(f"\nExtracting static features and calculating mean climatology across all available years...")
    
    static_data = {}
    dynamic_sums = {f: None for f in DYNAMIC_FEATURES}
    dynamic_counts = {f: None for f in DYNAMIC_FEATURES}
    nodata_val = None

    # Loop over all available years and months in the new FWI dataset
    feature_files = list(FEATURE_TIF_DIR.glob("*_new_fwi_with_fraction.tif"))
    
    if not feature_files:
        print(f"❌ No feature files found in {FEATURE_TIF_DIR}. Exiting.")
        return
        
    for tif_path in tqdm(feature_files, desc="Reading multi-year weather/FWI data"):
        with rio.open(tif_path) as src:
            if nodata_val is None: nodata_val = src.nodata
            
            # Read static features only on the first pass
            if not static_data:
                for name, b_idx in STATIC_FEATURES.items():
                    static_data[name] = src.read(b_idx).astype('float32')
            
            # Accumulate dynamic features for the all-time mean
            for name, b_idx in DYNAMIC_FEATURES.items():
                arr = src.read(b_idx).astype('float32')
                
                # Clean nodata
                if nodata_val is not None: arr[arr == nodata_val] = np.nan
                arr[arr < -9000] = np.nan
                
                valid_mask = ~np.isnan(arr)
                
                if dynamic_sums[name] is None:
                    dynamic_sums[name] = np.zeros_like(arr)
                    dynamic_counts[name] = np.zeros_like(arr, dtype=int)
                
                dynamic_sums[name][valid_mask] += arr[valid_mask]
                dynamic_counts[name][valid_mask] += 1

    # 3. Flatten and Clean the Data
    print("\nFlattening spatial data into a DataFrame...")
    
    df_dict = {"Burn_Count": cumulative_burn.flatten()}
    
    # Add Static
    for name in STATIC_FEATURES:
        df_dict[name] = static_data[name].flatten()
        
    # Add Dynamic (Calculate Mean safely avoiding division by zero)
    for name in DYNAMIC_FEATURES:
        mean_arr = np.divide(
            dynamic_sums[name], 
            dynamic_counts[name], 
            out=np.full_like(dynamic_sums[name], np.nan), 
            where=dynamic_counts[name] != 0
        )
        df_dict[name + " (Mean)"] = mean_arr.flatten()

    df = pd.DataFrame(df_dict)

    # Drop NoData/Ocean pixels
    initial_len = len(df)
    if nodata_val is not None:
        df = df[df["DEM"] != nodata_val]
    df = df[df["DEM"] > -500]  
    df = df.dropna()
    print(f"Filtered out NoData/Ocean. Valid land pixels: {len(df):,} (from {initial_len:,})")

    # 4. Create Categories & Apply Text Labels
    def categorize_burns(count):
        if count == 0: return "0: Never Burned"
        elif count <= 5: return "1-5 Times"
        elif count <= 10: return "6-10 Times"
        else: return "11+ Times"

    df["Burn_Category"] = df["Burn_Count"].apply(categorize_burns)
    category_order = ["0: Never Burned", "1-5 Times", "6-10 Times", "11+ Times"]

    # Map the integers to string descriptions
    df["LandCover_Label"] = df["LandCover_b1"].map(LAND_COVER_DICT).fillna("Unknown")

    print("\nPixel Distribution by Category:")
    print(df["Burn_Category"].value_counts())

    sns.set_theme(style="whitegrid")

    # 5. Plotting: Boxplots for STATIC Variables
    print("\nGenerating Static Diagnostic Boxplots...")
    static_vars = ["DEM", "Slope", "Aspect"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for i, var in enumerate(static_vars):
        sns.boxplot(data=df, x="Burn_Category", y=var, ax=axes[i], order=category_order, palette="Reds")
        axes[i].set_title(f"Distribution of {var} (New FWI)", fontweight='bold')
        axes[i].set_xlabel("")
        axes[i].set_ylabel(var)
        axes[i].tick_params(axis='x', rotation=15)

    plt.tight_layout()
    plt.savefig(OUT_DIR / "static_feature_boxplots_new_fwi.png", dpi=300)
    plt.close()

    # 6. Plotting: Boxplots for DYNAMIC Variables
    print("Generating Dynamic Diagnostic Boxplots (All-Time Means)...")
    dynamic_vars = [f"{name} (Mean)" for name in DYNAMIC_FEATURES.keys()]
    
    # Create a grid for the 11 dynamic variables (e.g., 3 rows, 4 columns)
    fig, axes = plt.subplots(3, 4, figsize=(24, 18))
    axes_flat = axes.flatten()

    for i, var in enumerate(dynamic_vars):
        sns.boxplot(data=df, x="Burn_Category", y=var, ax=axes_flat[i], order=category_order, palette="Reds")
        axes_flat[i].set_title(f"All-Time {var}", fontweight='bold')
        axes_flat[i].set_xlabel("")
        axes_flat[i].set_ylabel("")
        axes_flat[i].tick_params(axis='x', rotation=20)

    # Hide the empty 12th subplot
    axes_flat[-1].set_visible(False)

    plt.tight_layout()
    plt.savefig(OUT_DIR / "dynamic_feature_boxplots_new_fwi.png", dpi=300)
    plt.close()

    # 7. Plotting: Horizontal Bar Chart for Categorical Land Cover
    print("Generating Land Cover distribution...")
    plt.figure(figsize=(14, 12))
    
    # Filter to show only Never Burned vs 11+ Times for cleaner comparison
    df_lc = df[df["Burn_Category"].isin(["0: Never Burned", "11+ Times"])]
    
    sns.histplot(
        data=df_lc,
        y="LandCover_Label",  # Y-axis to make labels readable
        hue="Burn_Category",
        stat="probability",
        common_norm=False,
        multiple="dodge",
        palette=["#cccccc", "#cc0000"]
    )
    plt.title("Land Cover Proportions: 11+ Times vs Never Burned (New FWI)", fontsize=16, fontweight='bold')
    plt.ylabel("Land Cover Class")
    plt.xlabel("Proportion of Pixels in Category")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "landcover_b1_distribution_new_fwi.png", dpi=300)
    plt.close()

    print(f"\n✅ Pixel Autopsy complete. Plots saved to: {OUT_DIR}")

if __name__ == "__main__":
    main()

STARTING PIXEL AUTOPSY (NEW FWI - DYNAMIC THRESHOLDS)
Aggregating 22 years of predictions...


Summing burns: 100%|██████████| 22/22 [00:00<00:00, 54.96it/s]



Extracting static features and calculating mean climatology across all available years...


Reading multi-year weather/FWI data:   3%|▎         | 8/264 [00:10<05:30,  1.29s/it]

090

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Pixel Autopsy: Analyze Features of Frequently Burned Pixels (0.90 Threshold)
-- Computes the ALL-YEARS mean for dynamic weather and FWI features --
-- Maps numerical Land Cover (b1) values to descriptive labels --
-- Generates comprehensive diagnostic boxplots and bar charts --
"""

import os
import numpy as np
import pandas as pd
import rasterio as rio
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# Set non-interactive backend
plt.switch_backend('Agg')

# ============================
# CONFIG
# ============================

# Point to the 0.90 LOYO predictions
PRED_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_loyo_regularized_new_fwi/annual_binary_masks_090")

# Point to the new FWI feature directory
FEATURE_TIF_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_new_fwi_with_fraction")

# Output directory specifically for the new FWI 0.90 autopsy
OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_global_diagnostics/autopsy_new_fwi_090")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Feature Band Indices (1-indexed for Rasterio)
STATIC_FEATURES = {
    "DEM": 1,
    "Slope": 2,
    "Aspect": 3,
    "LandCover_b1": 4
}

DYNAMIC_FEATURES = {
    "Relative Humidity": 5,
    "Total Precip Sum": 6,
    "Temp 2m": 7,
    "Temp 2m Min": 8,
    "Temp 2m Max": 9,
    "Build Up Index": 10,
    "Drought Code": 11,
    "Duff Moisture Code": 12,
    "Fine Fuel Moisture Code": 13,
    "Fire Weather Index": 14,
    "Initial Fire Spread Index": 15
}

# Transcribed Land Cover Dictionary
LAND_COVER_DICT = {
    0: "Unclassified", 6: "Poplar forest", 7: "Oak forest", 10: "Pine forest",
    11: "Fir forest", 12: "Aspen forest", 13: "Hemlock forest", 14: "Spruce forest",
    15: "White Spruce", 16: "Black Spruce", 17: "Mixed forest", 18: "Larch forest",
    19: "Birch forest", 20: "Scotts Pine forest", 21: "Jack Pine forest",
    22: "Cedar forest", 23: "Maple", 24: "Siberian Pine", 25: "Linden",
    26: "Cedar Elfin Wood", 30: "Alpine shrubland", 31: "Riparian shrubland",
    32: "Other shrubland", 33: "Herbaceous", 35: "Sparsely Vegetated",
    50: "Bog", 51: "Fen", 52: "Marsh", 60: "Erect-shrub tundra",
    61: "Prostrate-shrub tundra", 62: "Shrub tundra", 63: "Graminoid tundra",
    64: "Wet-sedge tundra", 65: "Barren tundra", 70: "Developed",
    98: "Snow/Ice", 99: "Other"
}

# ============================
# MAIN
# ============================

def main():
    print("=" * 60)
    print("STARTING PIXEL AUTOPSY (NEW FWI - 0.90 THRESHOLD)")
    print("=" * 60)

    # 1. Accumulate Burn Counts
    # UPDATED glob pattern for 0.90 files
    pred_files = sorted(list(PRED_DIR.glob("loyo_regularized_new_fwi_annual_binary_*_0.90.tif")))
    
    if not pred_files:
        print(f"❌ No prediction files found in {PRED_DIR}. Exiting.")
        return

    # Extract year cleanly by removing the 0.90 suffix
    years = [int(f.name.replace('_0.90.tif', '').split('_')[-1]) for f in pred_files]
    print(f"Aggregating {len(pred_files)} years of predictions...")
    cumulative_burn = None
    
    for pred_path in tqdm(pred_files, desc="Summing burns"):
        with rio.open(pred_path) as src:
            mask = src.read(1).astype(np.int32) 
            if cumulative_burn is None:
                cumulative_burn = mask
            else:
                cumulative_burn += mask

    # 2. Extract Static Features & Accumulate Dynamic Features across ALL years
    print(f"\nExtracting static features and calculating mean climatology across all available years...")
    
    static_data = {}
    dynamic_sums = {f: None for f in DYNAMIC_FEATURES}
    dynamic_counts = {f: None for f in DYNAMIC_FEATURES}
    nodata_val = None

    # Loop over all available years and months in the new FWI dataset
    feature_files = list(FEATURE_TIF_DIR.glob("*_new_fwi_with_fraction.tif"))
    
    if not feature_files:
        print(f"❌ No feature files found in {FEATURE_TIF_DIR}. Exiting.")
        return
        
    for tif_path in tqdm(feature_files, desc="Reading multi-year weather/FWI data"):
        with rio.open(tif_path) as src:
            if nodata_val is None: nodata_val = src.nodata
            
            # Read static features only on the first pass
            if not static_data:
                for name, b_idx in STATIC_FEATURES.items():
                    static_data[name] = src.read(b_idx).astype('float32')
            
            # Accumulate dynamic features for the all-time mean
            for name, b_idx in DYNAMIC_FEATURES.items():
                arr = src.read(b_idx).astype('float32')
                
                # Clean nodata
                if nodata_val is not None: arr[arr == nodata_val] = np.nan
                arr[arr < -9000] = np.nan
                
                valid_mask = ~np.isnan(arr)
                
                if dynamic_sums[name] is None:
                    dynamic_sums[name] = np.zeros_like(arr)
                    dynamic_counts[name] = np.zeros_like(arr, dtype=int)
                
                dynamic_sums[name][valid_mask] += arr[valid_mask]
                dynamic_counts[name][valid_mask] += 1

    # 3. Flatten and Clean the Data
    print("\nFlattening spatial data into a DataFrame...")
    
    df_dict = {"Burn_Count": cumulative_burn.flatten()}
    
    # Add Static
    for name in STATIC_FEATURES:
        df_dict[name] = static_data[name].flatten()
        
    # Add Dynamic (Calculate Mean safely avoiding division by zero)
    for name in DYNAMIC_FEATURES:
        mean_arr = np.divide(
            dynamic_sums[name], 
            dynamic_counts[name], 
            out=np.full_like(dynamic_sums[name], np.nan), 
            where=dynamic_counts[name] != 0
        )
        df_dict[name + " (Mean)"] = mean_arr.flatten()

    df = pd.DataFrame(df_dict)

    # Drop NoData/Ocean pixels
    initial_len = len(df)
    if nodata_val is not None:
        df = df[df["DEM"] != nodata_val]
    df = df[df["DEM"] > -500]  
    df = df.dropna()
    print(f"Filtered out NoData/Ocean. Valid land pixels: {len(df):,} (from {initial_len:,})")

    # 4. Create Categories & Apply Text Labels
    def categorize_burns(count):
        if count == 0: return "0: Never Burned"
        elif count <= 5: return "1-5 Times"
        elif count <= 10: return "6-10 Times"
        else: return "11+ Times"

    df["Burn_Category"] = df["Burn_Count"].apply(categorize_burns)
    category_order = ["0: Never Burned", "1-5 Times", "6-10 Times", "11+ Times"]

    # Map the integers to string descriptions
    df["LandCover_Label"] = df["LandCover_b1"].map(LAND_COVER_DICT).fillna("Unknown")

    print("\nPixel Distribution by Category:")
    print(df["Burn_Category"].value_counts())

    sns.set_theme(style="whitegrid")

    # 5. Plotting: Boxplots for STATIC Variables
    print("\nGenerating Static Diagnostic Boxplots...")
    static_vars = ["DEM", "Slope", "Aspect"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for i, var in enumerate(static_vars):
        sns.boxplot(data=df, x="Burn_Category", y=var, ax=axes[i], order=category_order, palette="Reds")
        axes[i].set_title(f"Distribution of {var} (New FWI - 0.90)", fontweight='bold')
        axes[i].set_xlabel("")
        axes[i].set_ylabel(var)
        axes[i].tick_params(axis='x', rotation=15)

    plt.tight_layout()
    plt.savefig(OUT_DIR / "static_feature_boxplots_new_fwi_090.png", dpi=300)
    plt.close()

    # 6. Plotting: Boxplots for DYNAMIC Variables
    print("Generating Dynamic Diagnostic Boxplots (All-Time Means)...")
    dynamic_vars = [f"{name} (Mean)" for name in DYNAMIC_FEATURES.keys()]
    
    # Create a grid for the 11 dynamic variables (e.g., 3 rows, 4 columns)
    fig, axes = plt.subplots(3, 4, figsize=(24, 18))
    axes_flat = axes.flatten()

    for i, var in enumerate(dynamic_vars):
        sns.boxplot(data=df, x="Burn_Category", y=var, ax=axes_flat[i], order=category_order, palette="Reds")
        axes_flat[i].set_title(f"All-Time {var}", fontweight='bold')
        axes_flat[i].set_xlabel("")
        axes_flat[i].set_ylabel("")
        axes_flat[i].tick_params(axis='x', rotation=20)

    # Hide the empty 12th subplot
    axes_flat[-1].set_visible(False)

    plt.tight_layout()
    plt.savefig(OUT_DIR / "dynamic_feature_boxplots_new_fwi_090.png", dpi=300)
    plt.close()

    # 7. Plotting: Horizontal Bar Chart for Categorical Land Cover
    print("Generating Land Cover distribution...")
    plt.figure(figsize=(14, 12))
    
    # Filter to show only Never Burned vs 11+ Times for cleaner comparison
    df_lc = df[df["Burn_Category"].isin(["0: Never Burned", "11+ Times"])]
    
    sns.histplot(
        data=df_lc,
        y="LandCover_Label",  # Y-axis to make labels readable
        hue="Burn_Category",
        stat="probability",
        common_norm=False,
        multiple="dodge",
        palette=["#cccccc", "#cc0000"]
    )
    plt.title("Land Cover Proportions: 11+ Times vs Never Burned (New FWI - 0.90)", fontsize=16, fontweight='bold')
    plt.ylabel("Land Cover Class")
    plt.xlabel("Proportion of Pixels in Category")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "landcover_b1_distribution_new_fwi_090.png", dpi=300)
    plt.close()

    print(f"\n✅ Pixel Autopsy complete. Plots saved to: {OUT_DIR}")

if __name__ == "__main__":
    main()

STARTING PIXEL AUTOPSY (NEW FWI - 0.90 THRESHOLD)
Aggregating 22 years of predictions...


Summing burns: 100%|██████████| 22/22 [00:00<00:00, 58.86it/s]



Extracting static features and calculating mean climatology across all available years...


Reading multi-year weather/FWI data: 100%|██████████| 264/264 [05:32<00:00,  1.26s/it]



Flattening spatial data into a DataFrame...
Filtered out NoData/Ocean. Valid land pixels: 1,443,428 (from 4,273,642)

Pixel Distribution by Category:
0: Never Burned    1411831
1-5 Times            30791
6-10 Times             775
11+ Times               31
Name: Burn_Category, dtype: int64

Generating Static Diagnostic Boxplots...


/explore/nobackup/people/spotter5/temp_dir/ipykernel_3233183/1219903802.py:198: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="Burn_Category", y=var, ax=axes[i], order=category_order, palette="Reds")
/explore/nobackup/people/spotter5/temp_dir/ipykernel_3233183/1219903802.py:198: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="Burn_Category", y=var, ax=axes[i], order=category_order, palette="Reds")
/explore/nobackup/people/spotter5/temp_dir/ipykernel_3233183/1219903802.py:198: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot

Generating Dynamic Diagnostic Boxplots (All-Time Means)...


/explore/nobackup/people/spotter5/temp_dir/ipykernel_3233183/1219903802.py:217: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="Burn_Category", y=var, ax=axes_flat[i], order=category_order, palette="Reds")
/explore/nobackup/people/spotter5/temp_dir/ipykernel_3233183/1219903802.py:217: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="Burn_Category", y=var, ax=axes_flat[i], order=category_order, palette="Reds")
/explore/nobackup/people/spotter5/temp_dir/ipykernel_3233183/1219903802.py:217: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  s

Generating Land Cover distribution...

✅ Pixel Autopsy complete. Plots saved to: /explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_global_diagnostics/autopsy_new_fwi_090


In [6]:
't'

't'

Plot maps

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Generate Annual Summer (Jun-Aug) Mean Maps for Dynamic Predictors.
-- Calculates consistent global color stretches (vmin/vmax) across all years --
-- Generates a 3x4 multi-panel map for each year over a gray shapefile --
"""

import os
import numpy as np
import rasterio as rio
import geopandas as gpd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pathlib import Path
from tqdm import tqdm

# Set non-interactive backend
plt.switch_backend('Agg')

# ============================
# CONFIG
# ============================

FEATURE_TIF_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_with_fraction")
GRID_SHP_PATH = Path("/explore/nobackup/people/spotter5/cnn_mapping/raw_files/pred_grid_clip_small.shp")

OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/comparison_plots_dynamic_summer_means")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SUMMER_MONTHS = [6, 7, 8]

# Feature Band Indices (1-indexed for Rasterio)
DYNAMIC_FEATURES = {
    "Relative Humidity": 5,
    "Total Precip Sum": 6,
    "Temp 2m": 7,
    "Temp 2m Min": 8,
    "Temp 2m Max": 9,
    "Build Up Index": 10,
    "Drought Code": 11,
    "Duff Moisture Code": 12,
    "Fine Fuel Moisture Code": 13,
    "Fire Weather Index": 14,
    "Initial Fire Spread Index": 15
}

# Thematic Colormaps for different variable types
COLORMAPS = {
    "Relative Humidity": "YlGnBu",
    "Total Precip Sum": "Blues",
    "Temp 2m": "coolwarm",
    "Temp 2m Min": "coolwarm",
    "Temp 2m Max": "coolwarm",
    "Build Up Index": "YlOrRd",
    "Drought Code": "YlOrRd",
    "Duff Moisture Code": "YlOrRd",
    "Fine Fuel Moisture Code": "YlOrRd",
    "Fire Weather Index": "inferno",
    "Initial Fire Spread Index": "magma"
}

# Map background colors
FIG_BG_COLOR = 'white'
SHP_COLOR = '#d3d3d3'
LINE_COLOR = '#808080'

# ============================
# HELPER FUNCTIONS
# ============================

def get_summer_means_for_year(year, nodata_val=None, profile=None):
    """Calculates the mean of the dynamic features for months 6, 7, 8 of a given year."""
    sums = {f: None for f in DYNAMIC_FEATURES}
    counts = {f: None for f in DYNAMIC_FEATURES}
    
    months_found = 0
    for month in SUMMER_MONTHS:
        tif_path = FEATURE_TIF_DIR / f"cems_e5l_firecci_{year}_{month}_with_fraction.tif"
        if not tif_path.exists():
            continue
            
        months_found += 1
        with rio.open(tif_path) as src:
            if nodata_val is None: nodata_val = src.nodata
            if profile is None: profile = src.profile.copy()
            
            for name, b_idx in DYNAMIC_FEATURES.items():
                arr = src.read(b_idx).astype('float32')
                
                # Clean nodata/outliers
                if nodata_val is not None: arr[arr == nodata_val] = np.nan
                arr[arr < -9000] = np.nan
                
                valid_mask = ~np.isnan(arr)
                
                if sums[name] is None:
                    sums[name] = np.zeros_like(arr)
                    counts[name] = np.zeros_like(arr, dtype=int)
                
                sums[name][valid_mask] += arr[valid_mask]
                counts[name][valid_mask] += 1

    if months_found == 0:
        return None, None
        
    means = {}
    for name in DYNAMIC_FEATURES:
        means[name] = np.divide(
            sums[name], counts[name], 
            out=np.full_like(sums[name], np.nan), 
            where=counts[name] > 0
        )
        
    return means, profile

# ============================
# MAIN
# ============================

def main():
    print("=" * 60)
    print("STARTING DYNAMIC PREDICTOR SUMMER MAPPING")
    print("=" * 60)

    # 1. Identify all available years
    all_tifs = list(FEATURE_TIF_DIR.glob("cems_e5l_firecci_*_6_with_fraction.tif")) # Look for June files
    years = sorted(list(set([int(f.stem.split('_')[3]) for f in all_tifs])))
    print(f"Found {len(years)} years with summer data: {years}")

    if not years:
        print("No summer data found. Exiting.")
        return

    # 2. Load Shapefile Context
    print(f"Loading Grid Context: {GRID_SHP_PATH}")
    gdf_grid = gpd.read_file(GRID_SHP_PATH)
    
    # 3. PASS 1: Determine Global Color Scales (vmin, vmax)
    print("\nPass 1/2: Calculating robust global color scales across all years...")
    global_vmin = {f: np.inf for f in DYNAMIC_FEATURES}
    global_vmax = {f: -np.inf for f in DYNAMIC_FEATURES}
    
    raster_crs = None
    raster_bounds = None
    
    for year in tqdm(years, desc="Scanning years for min/max"):
        means, profile = get_summer_means_for_year(year)
        if means is None: continue
        
        if raster_crs is None:
            raster_crs = profile['crs']
            
            # Reconstruct bounds from transform and shape
            height, width = profile['height'], profile['width']
            transform = profile['transform']
            left, top = transform * (0, 0)
            right, bottom = transform * (width, height)
            raster_bounds = (left, right, bottom, top)

        for feat in DYNAMIC_FEATURES:
            arr = means[feat]
            # Use 2nd and 98th percentiles to drop crazy outliers from the scale
            if np.any(~np.isnan(arr)):
                p2 = np.nanpercentile(arr, 2)
                p98 = np.nanpercentile(arr, 98)
                global_vmin[feat] = min(global_vmin[feat], p2)
                global_vmax[feat] = max(global_vmax[feat], p98)

    print("\nLocked Global Color Scales:")
    for feat in DYNAMIC_FEATURES:
        print(f"  {feat}: {global_vmin[feat]:.2f} to {global_vmax[feat]:.2f}")

    # Reproject Shapefile to raster CRS
    if gdf_grid.crs != raster_crs:
        gdf_grid_proj = gdf_grid.to_crs(raster_crs)
    else:
        gdf_grid_proj = gdf_grid
        
    minx, miny, maxx, maxy = gdf_grid_proj.total_bounds
    extent = [raster_bounds[0], raster_bounds[1], raster_bounds[2], raster_bounds[3]]

    # 4. PASS 2: Generate Multi-Panel Maps Per Year
    print("\nPass 2/2: Generating annual multi-panel maps...")
    
    for year in tqdm(years, desc="Plotting Maps"):
        means, _ = get_summer_means_for_year(year)
        if means is None: continue

        # Create 3x4 grid for 11 features
        fig, axes = plt.subplots(3, 4, figsize=(24, 18), facecolor=FIG_BG_COLOR)
        axes_flat = axes.flatten()

        for i, feat in enumerate(DYNAMIC_FEATURES.keys()):
            ax = axes_flat[i]
            ax.set_facecolor(FIG_BG_COLOR)

            # A. Plot Shapefile Context
            gdf_grid_proj.plot(
                ax=ax, facecolor=SHP_COLOR, edgecolor=LINE_COLOR, 
                linewidth=0.5, zorder=1
            )

            # B. Plot Predictor Data
            cmap = COLORMAPS.get(feat, "viridis")
            
            im = ax.imshow(
                means[feat], 
                extent=extent, 
                cmap=cmap, 
                vmin=global_vmin[feat], 
                vmax=global_vmax[feat], 
                interpolation='none', 
                zorder=2
            )

            # C. Map Formatting
            ax.set_title(feat, fontsize=16, fontweight='bold')
            ax.set_xlim(minx, maxx)
            ax.set_ylim(miny, maxy)
            
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

            # D. Add Individual Colorbar
            divider = make_axes_locatable(ax)
            cax = divider.append_axes("right", size="5%", pad=0.1)
            cbar = fig.colorbar(im, cax=cax)
            cbar.ax.tick_params(labelsize=10)

        # Hide the 12th empty subplot
        axes_flat[-1].set_visible(False)

        # Main Title
        plt.suptitle(f"Summer (Jun-Aug) Mean Predictors: {year}", fontsize=24, fontweight='bold', y=1.02)
        
        plt.tight_layout()
        
        out_path = OUT_DIR / f"summer_dynamic_predictors_{year}.png"
        plt.savefig(out_path, dpi=200, bbox_inches='tight', facecolor=FIG_BG_COLOR)
        plt.close()

    print(f"\n✅ All annual maps generated and saved to: {OUT_DIR}")

if __name__ == "__main__":
    main()

STARTING DYNAMIC PREDICTOR SUMMER MAPPING
Found 22 years with summer data: [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
Loading Grid Context: /explore/nobackup/people/spotter5/cnn_mapping/raw_files/pred_grid_clip_small.shp

Pass 1/2: Calculating robust global color scales across all years...


Scanning years for min/max: 100%|██████████| 22/22 [01:45<00:00,  4.81s/it]



Locked Global Color Scales:
  Relative Humidity: 55.52 to 87.86
  Total Precip Sum: 0.00 to 0.00
  Temp 2m: 272.67 to 292.63
  Temp 2m Min: 265.20 to 282.63
  Temp 2m Max: 278.05 to 306.69
  Build Up Index: 0.79 to 100.97
  Drought Code: 38.39 to 749.51
  Duff Moisture Code: 0.39 to 64.08
  Fine Fuel Moisture Code: 44.68 to 83.77
  Fire Weather Index: 0.18 to 20.52
  Initial Fire Spread Index: 0.44 to 6.06

Pass 2/2: Generating annual multi-panel maps...


Plotting Maps: 100%|██████████| 22/22 [08:52<00:00, 24.19s/it]


✅ All annual maps generated and saved to: /explore/nobackup/people/spotter5/clelland_fire_ml/comparison_plots_dynamic_summer_means


New fwi

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Generate Annual Summer (Jun-Aug) Mean Maps for Dynamic Predictors (NEW FWI).
-- Calculates consistent global color stretches (vmin/vmax) across all years --
-- Generates a 3x4 multi-panel map for each year over a gray shapefile --
"""

import os
import numpy as np
import rasterio as rio
import geopandas as gpd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pathlib import Path
from tqdm import tqdm

# Set non-interactive backend
plt.switch_backend('Agg')

# ============================
# CONFIG
# ============================

# UPDATED: Point to the new_fwi directory
FEATURE_TIF_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_new_fwi_with_fraction")
GRID_SHP_PATH = Path("/explore/nobackup/people/spotter5/cnn_mapping/raw_files/pred_grid_clip_small.shp")

# UPDATED: Output to a new_fwi specific folder
OUT_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/comparison_plots_dynamic_summer_means_new_fwi")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SUMMER_MONTHS = [6, 7, 8]

# Feature Band Indices (1-indexed for Rasterio)
# Assumes the stacking order: [Primary Bands] + [New FWI Bands]
DYNAMIC_FEATURES = {
    "Relative Humidity": 5,
    "Total Precip Sum": 6,
    "Temp 2m": 7,
    "Temp 2m Min": 8,
    "Temp 2m Max": 9,
    "Build Up Index": 11,        # Indices adjusted based on your stack logic
    "Drought Code": 12,
    "Duff Moisture Code": 13,
    "Fine Fuel Moisture Code": 14,
    "Fire Weather Index": 15,
    "Initial Fire Spread Index": 16
}

# Thematic Colormaps for different variable types
COLORMAPS = {
    "Relative Humidity": "YlGnBu",
    "Total Precip Sum": "Blues",
    "Temp 2m": "coolwarm",
    "Temp 2m Min": "coolwarm",
    "Temp 2m Max": "coolwarm",
    "Build Up Index": "YlOrRd",
    "Drought Code": "YlOrRd",
    "Duff Moisture Code": "YlOrRd",
    "Fine Fuel Moisture Code": "YlOrRd",
    "Fire Weather Index": "inferno",
    "Initial Fire Spread Index": "magma"
}

# Map background colors
FIG_BG_COLOR = 'white'
SHP_COLOR = '#d3d3d3'
LINE_COLOR = '#808080'

# ============================
# HELPER FUNCTIONS
# ============================

def get_summer_means_for_year(year, nodata_val=None, profile=None):
    """Calculates the mean of the dynamic features for months 6, 7, 8 of a given year."""
    sums = {f: None for f in DYNAMIC_FEATURES}
    counts = {f: None for f in DYNAMIC_FEATURES}
    
    months_found = 0
    for month in SUMMER_MONTHS:
        # UPDATED: Matches the new filename suffix
        tif_path = FEATURE_TIF_DIR / f"cems_e5l_firecci_{year}_{month}_new_fwi_with_fraction.tif"
        if not tif_path.exists():
            continue
            
        months_found += 1
        with rio.open(tif_path) as src:
            if nodata_val is None: nodata_val = src.nodata
            if profile is None: profile = src.profile.copy()
            
            for name, b_idx in DYNAMIC_FEATURES.items():
                try:
                    arr = src.read(b_idx).astype('float32')
                    
                    # Clean nodata/outliers
                    if nodata_val is not None: arr[arr == nodata_val] = np.nan
                    arr[arr < -9000] = np.nan
                    
                    valid_mask = ~np.isnan(arr)
                    
                    if sums[name] is None:
                        sums[name] = np.zeros_like(arr)
                        counts[name] = np.zeros_like(arr, dtype=int)
                    
                    sums[name][valid_mask] += arr[valid_mask]
                    counts[name][valid_mask] += 1
                except IndexError:
                    print(f"[WARN] Band {b_idx} ({name}) missing in {tif_path.name}")

    if months_found == 0:
        return None, None
        
    means = {}
    for name in DYNAMIC_FEATURES:
        if sums[name] is not None:
            means[name] = np.divide(
                sums[name], counts[name], 
                out=np.full_like(sums[name], np.nan), 
                where=counts[name] > 0
            )
        else:
            means[name] = None
        
    return means, profile

# ============================
# MAIN
# ============================

def main():
    print("=" * 60)
    print("STARTING DYNAMIC PREDICTOR SUMMER MAPPING (NEW FWI)")
    print("=" * 60)

    # 1. Identify all available years
    # UPDATED: Search for the new suffix
    all_tifs = list(FEATURE_TIF_DIR.glob("cems_e5l_firecci_*_6_new_fwi_with_fraction.tif"))
    years = sorted(list(set([int(f.name.split('_')[3]) for f in all_tifs])))
    print(f"Found {len(years)} years with summer data: {years}")

    if not years:
        print(f"No summer data found in {FEATURE_TIF_DIR}. Exiting.")
        return

    # 2. Load Shapefile Context
    print(f"Loading Grid Context: {GRID_SHP_PATH}")
    gdf_grid = gpd.read_file(GRID_SHP_PATH)
    
    # 3. PASS 1: Determine Global Color Scales (vmin, vmax)
    print("\nPass 1/2: Calculating robust global color scales across all years...")
    global_vmin = {f: np.inf for f in DYNAMIC_FEATURES}
    global_vmax = {f: -np.inf for f in DYNAMIC_FEATURES}
    
    raster_crs = None
    raster_bounds = None
    
    for year in tqdm(years, desc="Scanning years for min/max"):
        means, profile = get_summer_means_for_year(year)
        if means is None: continue
        
        if raster_crs is None:
            raster_crs = profile['crs']
            height, width = profile['height'], profile['width']
            transform = profile['transform']
            left, top = transform * (0, 0)
            right, bottom = transform * (width, height)
            raster_bounds = (left, right, bottom, top)

        for feat in DYNAMIC_FEATURES:
            arr = means[feat]
            if arr is not None and np.any(~np.isnan(arr)):
                p2 = np.nanpercentile(arr, 2)
                p98 = np.nanpercentile(arr, 98)
                global_vmin[feat] = min(global_vmin[feat], p2)
                global_vmax[feat] = max(global_vmax[feat], p98)

    print("\nLocked Global Color Scales:")
    for feat in DYNAMIC_FEATURES:
        if global_vmin[feat] == np.inf: continue
        print(f"  {feat}: {global_vmin[feat]:.2f} to {global_vmax[feat]:.2f}")

    if gdf_grid.crs != raster_crs:
        gdf_grid_proj = gdf_grid.to_crs(raster_crs)
    else:
        gdf_grid_proj = gdf_grid
        
    minx, miny, maxx, maxy = gdf_grid_proj.total_bounds
    extent = [raster_bounds[0], raster_bounds[1], raster_bounds[2], raster_bounds[3]]

    # 4. PASS 2: Generate Multi-Panel Maps Per Year
    print("\nPass 2/2: Generating annual multi-panel maps...")
    
    for year in tqdm(years, desc="Plotting Maps"):
        means, _ = get_summer_means_for_year(year)
        if means is None: continue

        fig, axes = plt.subplots(3, 4, figsize=(24, 18), facecolor=FIG_BG_COLOR)
        axes_flat = axes.flatten()

        for i, feat in enumerate(DYNAMIC_FEATURES.keys()):
            ax = axes_flat[i]
            ax.set_facecolor(FIG_BG_COLOR)

            # A. Plot Shapefile Context
            gdf_grid_proj.plot(
                ax=ax, facecolor=SHP_COLOR, edgecolor=LINE_COLOR, 
                linewidth=0.5, zorder=1
            )

            # B. Plot Predictor Data
            if means[feat] is not None:
                cmap = COLORMAPS.get(feat, "viridis")
                im = ax.imshow(
                    means[feat], 
                    extent=extent, 
                    cmap=cmap, 
                    vmin=global_vmin[feat], 
                    vmax=global_vmax[feat], 
                    interpolation='none', 
                    zorder=2
                )

                # D. Add Individual Colorbar
                divider = make_axes_locatable(ax)
                cax = divider.append_axes("right", size="5%", pad=0.1)
                cbar = fig.colorbar(im, cax=cax)
                cbar.ax.tick_params(labelsize=10)

            # C. Map Formatting
            ax.set_title(feat, fontsize=16, fontweight='bold')
            ax.set_xlim(minx, maxx)
            ax.set_ylim(miny, maxy)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

        axes_flat[-1].set_visible(False)

        # UPDATED: Title reflects new FWI
        plt.suptitle(f"Summer (Jun-Aug) Mean Predictors (New FWI): {year}", fontsize=24, fontweight='bold', y=1.02)
        
        plt.tight_layout()
        
        out_path = OUT_DIR / f"summer_dynamic_predictors_new_fwi_{year}.png"
        plt.savefig(out_path, dpi=200, bbox_inches='tight', facecolor=FIG_BG_COLOR)
        plt.close()

    print(f"\n✅ All annual maps generated and saved to: {OUT_DIR}")

if __name__ == "__main__":
    main()

STARTING DYNAMIC PREDICTOR SUMMER MAPPING (NEW FWI)
Found 22 years with summer data: [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
Loading Grid Context: /explore/nobackup/people/spotter5/cnn_mapping/raw_files/pred_grid_clip_small.shp

Pass 1/2: Calculating robust global color scales across all years...


Scanning years for min/max: 100%|██████████| 22/22 [01:44<00:00,  4.73s/it]



Locked Global Color Scales:
  Relative Humidity: 55.52 to 87.86
  Total Precip Sum: 0.00 to 0.00
  Temp 2m: 272.67 to 292.63
  Temp 2m Min: 265.20 to 282.63
  Temp 2m Max: 278.05 to 306.69
  Build Up Index: 0.71 to 100.92
  Drought Code: 38.32 to 751.55
  Duff Moisture Code: 0.35 to 64.08
  Fine Fuel Moisture Code: 44.58 to 83.81
  Fire Weather Index: 0.18 to 20.57
  Initial Fire Spread Index: 0.44 to 6.08

Pass 2/2: Generating annual multi-panel maps...


Plotting Maps: 100%|██████████| 22/22 [09:39<00:00, 26.35s/it]


✅ All annual maps generated and saved to: /explore/nobackup/people/spotter5/clelland_fire_ml/comparison_plots_dynamic_summer_means_new_fwi


In [ ]:
't'

In [5]:
import rasterio as rio
import numpy as np
from pathlib import Path

# ============================
# CONFIG
# ============================
TIF_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/ml_training/xgb_loyo_regularized_new_fwi/prediction_tifs")
YEAR = 2003

def check_annual_count(year, tif_dir):
    # Find all monthly binary files for the year
    # Pattern: loyo_pred_bin_2003_01.tif, etc.
    tif_paths = sorted(list(tif_dir.glob(f"loyo_pred_bin_{year}_*.tif")))
    
    if not tif_paths:
        print(f"No TIFFs found for year {year} in {tif_dir}")
        return

    print(f"Reading {len(tif_paths)} monthly TIFFs for {year}...")
    
    annual_stack = []
    
    for p in tif_paths:
        with rio.open(p) as src:
            data = src.read(1)
            # Mask out the NoData values (255) to 0 so they don't interfere with max()
            # We only want to count actual '1' (burned) predictions
            clean_data = np.where(data == 1, 1, 0)
            annual_stack.append(clean_data)
            
    # Stack months and take the max across the time axis
    # This represents: If pixel was 1 in any month, it is 1 for the year
    annual_max = np.max(np.stack(annual_stack), axis=0)
    
    total_burned_pixels = np.sum(annual_max)
    
    print("-" * 30)
    print(f"ANNUAL RESULTS FOR {year}")
    print("-" * 30)
    print(f"Total Unique Burned Pixels: {total_burned_pixels:,}")
    print(f"Training Log Expected:      1,700")
    print("-" * 30)

if __name__ == "__main__":
    check_annual_count(YEAR, TIF_DIR)

Reading 12 monthly TIFFs for 2003...
------------------------------
ANNUAL RESULTS FOR 2003
------------------------------
Total Unique Burned Pixels: 1,697
Training Log Expected:      1,700
------------------------------


In [4]:
import rasterio as rio
from pathlib import Path

# Path to your source TIFFs
TIF_SOURCE_DIR = Path("/explore/nobackup/people/spotter5/clelland_fire_ml/training_e5l_cems_firecci_new_fwi_with_fraction")

# Check a sample month from 2003
sample_file = TIF_SOURCE_DIR / "cems_e5l_firecci_2003_7_new_fwi_with_fraction.tif"

if not sample_file.exists():
    print(f"File not found: {sample_file}")
else:
    with rio.open(sample_file) as src:
        print(f"--- Metadata for: {sample_file.name} ---")
        print(f"CRS:        {src.crs}")
        print(f"Transform:  {src.transform}")
        print(f"Bounds:     {src.bounds}")
        print(f"Width/Height: {src.width} x {src.height}")
        print(f"NoData:     {src.nodata}")
        print(f"Band Count: {src.count}")
        
        # Check a small window of data from the first band to ensure it's readable
        data_sample = src.read(1, window=((0, 5), (0, 5)))
        print(f"\nTop-left 5x5 data sample (Band 1):\n{data_sample}")

--- Metadata for: cems_e5l_firecci_2003_7_new_fwi_with_fraction.tif ---
CRS:        EPSG:6931
Transform:  | 4000.00, 0.00,-4724000.00|
| 0.00,-4000.00, 4168000.00|
| 0.00, 0.00, 1.00|
Bounds:     BoundingBox(left=-4724000.0, bottom=-3484000.0, right=4212000.0, top=4168000.0)
Width/Height: 2234 x 1913
NoData:     None
Band Count: 16

Top-left 5x5 data sample (Band 1):
[[nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]]
